# 01 · Enhanced EDA — Washington State EV Population
> **Dataset**: WA State DOL Electric Vehicle Population Data — 130,443 records, ~99.8% WA registrations.

This notebook replaces the original single-notebook EDA with:
- Correct data-quality documentation (zero-range handling, Base MSRP issue)
- Statistical testing (chi-square, Mann-Whitney U, ANOVA)
- Interactive Plotly charts
- Market concentration analysis (HHI)
- Washington State choropleth map

In [2]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats
from pathlib import Path

from src.features import full_preprocessing_pipeline, RANGE_COL, MAKE_COL, YEAR_COL, EV_TYPE_COL, CAFV_COL, COUNTY_COL
from src import visualizations as viz

# ── Load & preprocess ──────────────────────────────────────────────────────
DATA_PATH = Path("../data/raw/Electric_Vehicle_Population_Data.csv")
df = full_preprocessing_pipeline(DATA_PATH)
print(f"Shape after preprocessing: {df.shape}")
df.head(3)


Dropped 199,320 duplicate rows


Shape after preprocessing: (81513, 19)


,County,City,State,Postal Code,Model Year,Make,Model,Electric Vehicle Type,Clean Alternative Fuel Vehicle (CAFV) Eligibility,Electric Range,Electric Utility,range_is_zero,cafv_label,cafv_code,is_bev,vehicle_age,make_market_share,median_range_by_make,is_tesla
0,King,Kirkland,WA,98034.0,2013,NISSAN,LEAF,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,75.0,PUGET SOUND ENERGY INC||CITY OF TACOMA - (WA),False,eligible,0,1,11,0.058641,84.0,0
1,Kitsap,Bainbridge Island,WA,98110.0,2020,TESLA,MODEL 3,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,308.0,PUGET SOUND ENERGY INC,False,eligible,0,1,4,0.178008,238.0,1
2,King,Seattle,WA,98144.0,2018,TESLA,MODEL 3,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,215.0,CITY OF SEATTLE - (WA)|CITY OF TACOMA - (WA),False,eligible,0,1,6,0.178008,238.0,1


## 1 · Data Quality Report

**Known data issues — documented explicitly:**

| Issue | Details |
|---|---|
| `Electric Range = 0` | ~41% of records. Represents vehicles whose EPA range has not been entered into the DOL database, not actually zero-range vehicles. Retained but flagged. |
| `Base MSRP = 0` | ~95% zeros. The column was collected but not populated for most registrations. Dropped. |
| `County/City` missing | ~3 records. Filled with 'Unknown'. |
| `Model` missing | ~222 records. Filled using Make name. |
| Geographic scope | 99.8% WA State — this is NOT a national dataset. |

In [3]:
# Data quality summary
print("=== Shape ===")
print(f"Rows: {len(df):,}  |  Columns: {df.shape[1]}")
print()

print("=== Zero-range breakdown ===")
zero = df["range_is_zero"].value_counts()
print(f"  Zero range : {zero[True]:,}  ({100*zero[True]/len(df):.1f}%)")
print(f"  Non-zero   : {zero[False]:,}  ({100*zero[False]/len(df):.1f}%)")
print()

print("=== State distribution (top 5) ===")
print(df["State"].value_counts().head())
print()

print("=== Missing values (remaining) ===")
print(df.isna().sum()[df.isna().sum() > 0])


=== Shape ===
Rows: 81,513  |  Columns: 19

=== Zero-range breakdown ===
  Zero range : 39,670  (48.7%)
  Non-zero   : 41,843  (51.3%)

=== State distribution (top 5) ===
State
WA    80816
CA      167
VA       88
FL       46
MD       45
Name: count, dtype: int64

=== Missing values (remaining) ===
Postal Code         12
Electric Range      12
Electric Utility    12
dtype: int64


## 2 · Core Distributions

In [4]:
viz.plot_model_year_distribution(df)

In [5]:
viz.plot_top_makes(df)

In [6]:
viz.plot_ev_type_over_time(df)

In [7]:
viz.plot_cafv_sunburst(df)

In [8]:
viz.plot_top_cities_treemap(df)

## 3 · Geographic Analysis — Washington State Choropleth

In [9]:
# County-level registration density
county_summary = (
    df[df["State"] == "WA"]
    .groupby(COUNTY_COL)
    .agg(
        ev_count=(RANGE_COL, "count"),
        pct_bev=("is_bev", "mean"),
        median_range=(RANGE_COL, lambda x: x[x > 0].median()),
    )
    .reset_index()
    .sort_values("ev_count", ascending=False)
)
print(county_summary.head(10).to_string(index=False))


   County  ev_count  pct_bev  median_range
     King     28952 0.652970          37.0
   Pierce     10088 0.674299          37.0
Snohomish      8044 0.660410          37.0
    Clark      4953 0.627977          37.0
  Spokane      4339 0.653226          38.0
 Thurston      3277 0.649069          37.0
   Kitsap      3101 0.654628          38.0
  Whatcom      2232 0.658602          38.0
   Benton      1515 0.621122          37.0
   Skagit      1385 0.664260          38.0


In [10]:
viz.plot_county_choropleth(df)

## 4 · Market Concentration (Herfindahl-Hirschman Index)

HHI = Σ(market_share_i²) × 10,000. Ranges: <1,500 = competitive, 1,500–2,500 = moderate, >2,500 = concentrated.

In [11]:
shares = df[MAKE_COL].value_counts(normalize=True)
hhi = (shares ** 2).sum() * 10_000
print(f"HHI (all makes): {hhi:.0f}")
print()
print("Interpretation:")
if hhi < 1500:
    print("  < 1,500 → Competitive market")
elif hhi < 2500:
    print("  1,500–2,500 → Moderately concentrated market")
else:
    print("  > 2,500 → Highly concentrated market (Tesla dominant)")

# Tesla-vs-rest breakdown
tesla_share = shares.get("TESLA", 0)
print(f"\nTesla market share: {tesla_share:.1%}")
print(f"Non-Tesla HHI (excluding Tesla): {((shares.drop('TESLA', errors='ignore') / (1 - tesla_share))**2).sum() * 10_000:.0f}")


HHI (all makes): 713

Interpretation:
  < 1,500 → Competitive market

Tesla market share: 17.8%
Non-Tesla HHI (excluding Tesla): 587


In [12]:
viz.plot_market_share_over_time(df)

## 5 · Statistical Tests

Replacing raw counts with inferential statistics.

In [13]:
from scipy.stats import chi2_contingency, mannwhitneyu, f_oneway

# ── Test 1: Chi-square — is BEV/PHEV split independent of county? ──────────
print("=" * 60)
print("TEST 1: Chi-square — BEV/PHEV vs County (Top 10 counties)")
print("=" * 60)
top_counties = df[COUNTY_COL].value_counts().head(10).index
sub = df[df[COUNTY_COL].isin(top_counties)]
contingency = pd.crosstab(sub[COUNTY_COL], sub["is_bev"])
chi2, p, dof, expected = chi2_contingency(contingency)
print(f"  χ² = {chi2:.2f},  df = {dof},  p = {p:.2e}")
print(f"  Verdict: {'Significant (p<0.05) — EV type varies by county' if p < 0.05 else 'Not significant'}")


TEST 1: Chi-square — BEV/PHEV vs County (Top 10 counties)
  χ² = 42.40,  df = 9,  p = 2.78e-06
  Verdict: Significant (p<0.05) — EV type varies by county


In [14]:
# ── Test 2: Mann-Whitney U — Tesla vs non-Tesla range ──────────────────────
print("=" * 60)
print("TEST 2: Mann-Whitney U — Tesla vs Non-Tesla Electric Range")
print("=" * 60)
df_nonzero = df[df[RANGE_COL] > 0]
tesla_range     = df_nonzero[df_nonzero[MAKE_COL] == "TESLA"][RANGE_COL]
non_tesla_range = df_nonzero[df_nonzero[MAKE_COL] != "TESLA"][RANGE_COL]
u_stat, p_mw = mannwhitneyu(tesla_range, non_tesla_range, alternative="greater")
print(f"  Tesla median range   : {tesla_range.median():.0f} miles")
print(f"  Non-Tesla median range: {non_tesla_range.median():.0f} miles")
print(f"  Mann-Whitney U = {u_stat:.0f},  p = {p_mw:.2e}")
print(f"  Verdict: {'Tesla ranges significantly higher (p<0.05)' if p_mw < 0.05 else 'Not significant'}")


TEST 2: Mann-Whitney U — Tesla vs Non-Tesla Electric Range
  Tesla median range   : 238 miles
  Non-Tesla median range: 34 miles
  Mann-Whitney U = 201769765,  p = 0.00e+00
  Verdict: Tesla ranges significantly higher (p<0.05)


In [15]:
# ── Test 3: ANOVA — range across model years (2017–2023) ───────────────────
print("=" * 60)
print("TEST 3: One-way ANOVA — Electric Range across Model Years (2017–2023)")
print("=" * 60)
years = range(2017, 2024)
groups = [
    df_nonzero[df_nonzero[YEAR_COL] == yr][RANGE_COL].values
    for yr in years
]
f_stat, p_anova = f_oneway(*groups)
print(f"  F = {f_stat:.2f},  p = {p_anova:.2e}")
print(f"  Verdict: {'Significant difference in range across years (p<0.05)' if p_anova < 0.05 else 'Not significant'}")
print()
for yr, g in zip(years, groups):
    print(f"  {yr}: n={len(g):,}  median={np.median(g):.0f} miles")


TEST 3: One-way ANOVA — Electric Range across Model Years (2017–2023)
  F = 2165.27,  p = 0.00e+00
  Verdict: Significant difference in range across years (p<0.05)

  2017: n=3,340  median=81 miles
  2018: n=4,107  median=53 miles
  2019: n=3,677  median=150 miles
  2020: n=4,462  median=259 miles
  2021: n=2,701  median=25 miles
  2022: n=2,883  median=28 miles
  2023: n=3,290  median=32 miles


In [16]:
# ── Test 4: Chi-square — CAFV eligibility vs BEV/PHEV ─────────────────────
print("=" * 60)
print("TEST 4: Chi-square — CAFV Eligibility vs EV Type")
print("=" * 60)
ct = pd.crosstab(df["cafv_label"], df["is_bev"])
chi2_cafv, p_cafv, dof_cafv, _ = chi2_contingency(ct)
print(ct)
print(f"\n  χ² = {chi2_cafv:.2f},  df = {dof_cafv},  p = {p_cafv:.2e}")
print(f"  Verdict: {'Strong association (p<0.05) between EV type and CAFV eligibility' if p_cafv < 0.05 else 'Not significant'}")


TEST 4: Chi-square — CAFV Eligibility vs EV Type
is_bev            0      1
cafv_label                
eligible      14638  14016
not_eligible  13180      9
unknown           0  39670

  χ² = 49622.62,  df = 2,  p = 0.00e+00
  Verdict: Strong association (p<0.05) between EV type and CAFV eligibility


## 6 · Range Analysis

In [17]:
viz.plot_range_by_make(df)

In [18]:
viz.plot_range_progression(df)

In [19]:
# Range milestone: what % of recent BEVs (2020+) offer ≥300 miles?
recent_bev = df[(df["is_bev"] == 1) & (df[YEAR_COL] >= 2020) & (df[RANGE_COL] > 0)]
pct_300 = 100 * (recent_bev[RANGE_COL] >= 300).mean()
print(f"BEVs (2020+) with ≥300 mile range: {pct_300:.1f}%")


BEVs (2020+) with ≥300 mile range: 23.9%


## 7 · Utility Provider Analysis

In [20]:
# Utility concentration
utility_counts = df["Electric Utility"].value_counts().head(15)
fig = px.bar(
    utility_counts.reset_index(),
    x="count", y="Electric Utility",
    orientation="h",
    title="Top 15 Utility Providers by EV Registration Count",
    template="plotly_white",
    labels={"count": "EV Registrations", "Electric Utility": "Utility"},
    color="count",
    color_continuous_scale="Purples",
)
fig.update_layout(coloraxis_showscale=False, yaxis={"categoryorder": "total ascending"})
fig.show()
